In [ ]:
# Cell 1: Dependencies
#!pip install google-generativeai datasets pandas pyarrow huggingface_hub langid

# Cell 2: Imports
import pandas as pd
import numpy as np
from datasets import Dataset, load_dataset, DatasetDict
import os
import json
import time
import google.generativeai as genai
from datetime import datetime
import re
import langid
from collections import Counter
import hashlib
from huggingface_hub import login, upload_file, hf_hub_download

In [ ]:
# Cell 3: Deep Thinking Legal QA Configuration
DOMAIN_CONFIG = {
    'domain': 'Deep Thinking Legal QA Dataset',
    'target_language': 'en',  # or 'id'
    'generation_mode': 'legal_qa_deep_thinking',
    'thinking_style': 'iterative',  # 'direct', 'analytical', 'iterative'
    'require_deep_analysis': True,
}

# Legal QA Taxonomy (same as before)
LEGAL_QA_TAXONOMY = {
    "corporate_commercial_qa": [
        "Interpretation of company policies in hypothetical scenarios",
        "Corporate governance decision-making Q&A",
        "Compliance risk scenario questions",
        "Deal structure conflict resolution Q&A",
        "M&A due diligence problem-solving questions",
    ],
    "contract_negotiation_qa": [
        "Clause interpretation questions in synthetic contracts",
        "Contract comparison reasoning Q&A",
        "Redline revision scenario questions",
        "Negotiation strategy scenario Q&A",
        "Identifying contractual vulnerabilities",
    ],
    "litigation_dispute_qa": [
        "Identify potential claims and defenses from fact patterns",
        "Evidence evaluation and credibility Q&A",
        "Settlement strategy reasoning questions",
        "Outcome prediction from scenario facts",
        "Trial strategy optimization Q&A",
    ],
    "legal_analysis_qa": [
        "Scenario-based advisory questions",
        "Risk assessment reasoning Q&A",
        "Ethical/legal dilemma scenario questions",
        "Policy compliance advisory Q&A",
        "Strategic decision-making in complex cases",
    ],
    "regulatory_compliance_qa": [
        "Compliance checklist validation Q&A",
        "Multi-jurisdiction regulatory scenario Q&A",
        "Regulatory risk mitigation Q&A",
        "Compliance failure scenario questions",
    ],
    "employment_hr_qa": [
        "Whistleblower complaint scenario Q&A",
        "Labor dispute reasoning Q&A",
        "Workplace investigation reasoning exercises",
        "Disciplinary action reasoning questions",
    ],
    "meta_legal_reasoning_qa": [
        "Negotiation strategy scenario Q&A",
        "Judge / jury prediction scenario questions",
        "Corporate crisis strategy reasoning Q&A",
        "Predictive litigation outcome exercises",
        "Strategic evidence and persuasion scenario questions",
    ]
}

# Deep thinking requirements
THINKING_REQUIREMENTS = {
    'min_thinking_words': 600,  # Minimum words for thinking process
    'min_answer_words': 150,    # Shorter direct answers
    'max_answer_words': 300,
    'required_iterations': 3,   # Minimum reconsideration loops
}

# System prompt for deep thinking
SYSTEM_PROMPT = """You are a senior legal advisor with deep analytical capabilities. When analyzing legal questions, you engage in extensive iterative thinking - considering multiple perspectives, checking assumptions, reconsidering positions, and validating conclusions before providing concise, well-reasoned advice."""

# Deep thinking QA generation prompt
DEEP_THINKING_QA_PROMPT = """Generate a legal question-answer pair with DEEP ITERATIVE THINKING:

TOPIC: {qa_topic}
CATEGORY: {category}

Create a complex legal scenario question that requires deep analysis, then provide your answer in TWO parts:

PART 1 - THINKING PROCESS (MINIMUM 600 WORDS):
Show your complete iterative reasoning with multiple reconsideration loops:

INITIAL ANALYSIS:
- What are the key facts and their legal significance?
- What legal issues are immediately apparent?
- What is my initial impression?

DEEPER EXAMINATION - ITERATION 1:
- Wait, what assumptions am I making?
- Are there alternative interpretations of the facts?
- What legal principles might apply differently?
- What am I potentially missing?

CROSS-CHECKING - ITERATION 2:
- Let me verify my reasoning against established precedents
- Are there counter-arguments I haven't considered?
- What would opposing counsel argue?
- How might a judge view this differently?

RISK ASSESSMENT - ITERATION 3:
- What are the practical implications of each interpretation?
- What are the best-case and worst-case scenarios?
- What hidden risks or opportunities exist?
- How confident am I in each aspect of my analysis?

SYNTHESIS AND VALIDATION:
- Reconsidering everything, what is the most defensible position?
- Where is my analysis strongest? Where is it weakest?
- What would I advise a client with high confidence vs. uncertainty?
- Final check: Am I missing anything critical?

PART 2 - DIRECT ANSWER (150-250 WORDS):
Concise, actionable legal advice based on the thinking above.

FORMAT:

Question: [Complex legal scenario requiring deep analysis, 100-200 words]

Thinking: [Extensive iterative analysis with multiple reconsideration loops, MINIMUM 600 words showing all reasoning stages]

Answer: [Concise, direct legal advice, 150-250 words]

Generate the complete deep-thinking QA:"""

# Alternative deep thinking prompt (for safety filter retry)
ALTERNATIVE_DEEP_THINKING_PROMPT = """Create a legal advisory training example with comprehensive analytical reasoning:

TOPIC: {qa_topic}

For legal education, generate:

1. A complex scenario question (100-200 words)

2. Extended analytical process (600+ words) showing:
   - Initial assessment and issue spotting
   - Multiple rounds of reconsideration and refinement
   - Alternative interpretations and counter-arguments
   - Risk analysis from multiple angles
   - Validation of conclusions
   - Final synthesis

3. Practical advisory answer (150-250 words)

Use this structure:

Question: [scenario]

Thinking: 
STAGE 1 - INITIAL ASSESSMENT: [analysis]
STAGE 2 - RECONSIDERATION: [re-evaluation]
STAGE 3 - ALTERNATIVE PERSPECTIVES: [counter-arguments]
STAGE 4 - RISK MAPPING: [risk analysis]
STAGE 5 - FINAL VALIDATION: [conclusion check]

Answer: [practical advice]

Generate complete example:"""

In [ ]:
# Cell 4: Main Configuration
CONFIG = {
    #'gemini_api_key': '',
    'gemini_api_key': '',
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    
    'huggingface_token': '',
    'output_repository': 'Azzindani/Legal_QA_Syn',
    
    # Deep thinking settings
    'batch_size': 5,  # Fewer per batch due to length
    'model_name': 'gemini-2.5-flash',
    'temperature': 0.85,
    'max_output_tokens': 6000,  # Much higher for deep thinking
    
    # Quality requirements
    'enforce_min_thinking_length': True,
    'require_iterations': True,
    
    'target_language_confidence': 0.5,
    'progress_file': 'legal_qa_deep_thinking_progress.json'
}

In [ ]:
# Cell 5: Authentication
genai.configure(api_key=CONFIG['gemini_api_key'])
login(token=CONFIG['huggingface_token'])

In [ ]:
# Cell 6: Text Validator for Single-Shot Processing
class TextValidator:
    def __init__(self):
        self.max_length = DOMAIN_CONFIG['max_text_length']
        self.min_length = DOMAIN_CONFIG['min_text_length']
    
    def validate_and_prepare_text(self, text):
        """Validate and prepare text for single-shot processing"""
        if not text or not isinstance(text, str):
            return None
        
        text = text.strip()
        
        if len(text) < self.min_length:
            return None
        
        # Truncate if too long (preserve beginning and end)
        if len(text) > self.max_length:
            half_max = self.max_length // 2 - 100  # Leave room for separator
            text = text[:half_max] + "\n\n[...TRUNCATED...]\n\n" + text[-half_max:]
        
        return text

In [ ]:
# Cell 7: Complete Language Detection with Both Methods
class LanguageScorer:
    def __init__(self, target_language='en'):
        self.target_language = target_language
    
    def detect_language(self, text):
        """Detect language with fixed confidence calculation"""
        if not text or len(text.strip()) < 10:
            return 'unknown', 0.0
        
        try:
            lang, raw_confidence = langid.classify(text.strip())
            
            # Fix: langid returns NEGATIVE log probability
            import math
            
            if raw_confidence <= 0:
                normalized_confidence = math.exp(raw_confidence)
            else:
                normalized_confidence = 1.0
            
            normalized_confidence = max(0.0, min(1.0, normalized_confidence))
            
            return lang, normalized_confidence
            
        except Exception as e:
            return 'unknown', 0.0
    
    def score_language_accuracy(self, question, answer):
        """Score language for Q&A pairs - REQUIRED FOR QA GENERATOR"""
        if not question or not answer:
            return {
                'question_language': 'unknown',
                'question_confidence': 0.0,
                'answer_language': 'unknown',
                'answer_confidence': 0.0,
                'language_accuracy': 0.0,
                'passes_language_check': False,
            }
        
        q_lang, q_conf = self.detect_language(question)
        a_lang, a_conf = self.detect_language(answer)
        
        # Ensure valid floats
        q_conf = max(0.0, min(1.0, float(q_conf)))
        a_conf = max(0.0, min(1.0, float(a_conf)))
        
        scores = {
            'question_language': str(q_lang),
            'question_confidence': float(q_conf),
            'answer_language': str(a_lang),
            'answer_confidence': float(a_conf),
            'question_correct_language': q_lang == self.target_language,
            'answer_correct_language': a_lang == self.target_language,
            'language_accuracy': float((q_conf + a_conf) / 2),
            'passes_language_check': True  # Always pass for now
        }
        
        return scores
    
    def score_corpus_language(self, corpus_text):
        """Score language for corpus text - FOR CORPUS GENERATOR"""
        if not corpus_text or len(corpus_text.strip()) < 50:
            return {
                'detected_language': 'unknown',
                'average_confidence': 0.0,
                'language_consistent': False,
                'matches_target': False,
            }
        
        # Sample from different parts
        text_len = len(corpus_text)
        samples = []
        
        if text_len > 0:
            samples.append(corpus_text[:min(500, text_len)])
        if text_len > 1000:
            mid = text_len // 2
            samples.append(corpus_text[mid:min(mid+500, text_len)])
        if text_len > 500:
            samples.append(corpus_text[-500:])
        
        # Detect language for each sample
        detections = []
        for sample in samples:
            if len(sample.strip()) < 10:
                continue
            lang, conf = self.detect_language(sample)
            if lang != 'unknown' and conf > 0:
                detections.append((lang, conf))
        
        if not detections:
            return {
                'detected_language': 'unknown',
                'average_confidence': 0.0,
                'language_consistent': False,
                'matches_target': False,
            }
        
        # Calculate statistics
        languages = [d[0] for d in detections]
        confidences = [d[1] for d in detections]
        
        most_common_lang = max(set(languages), key=languages.count)
        avg_confidence = sum(confidences) / len(confidences)
        is_consistent = all(lang == most_common_lang for lang in languages)
        
        return {
            'detected_language': str(most_common_lang),
            'average_confidence': float(avg_confidence),
            'language_consistent': bool(is_consistent),
            'matches_target': bool(most_common_lang == self.target_language),
        }

In [ ]:
# Cell 8: Progress Manager (adapted for corpus)
class ProgressManager:
    def __init__(self):
        self.progress_data = {
            'processed_chunks': [],
            'current_row': 0,
            'total_chunks_processed': 0,
            'total_qa_pairs_created': 0,
            'request_count': 0,
            'start_time': None,
            'last_update': None,
            'errors': [],
            'statistics': {
                'avg_chunks_per_text': 0.0,
                'avg_qa_per_chunk': 0.0,
                'approach_stats': {'simple': 0, 'deep_thinking': 0, 'iterative_thinking': 0}
            }
        }
        self.load_progress()
    
    def load_progress(self):
        try:
            progress_path = hf_hub_download(
                repo_id=CONFIG['output_repository'],
                filename=CONFIG['progress_file'],
                repo_type="dataset"
            )
            with open(progress_path, 'r') as f:
                saved_progress = json.load(f)
                self.progress_data.update(saved_progress)
            print(f"Progress loaded: {self.progress_data['total_chunks_processed']} chunks processed")
        except Exception as e:
            print(f"No existing progress found, starting fresh: {e}")
            self.progress_data['start_time'] = datetime.now().isoformat()
    
    def save_progress(self):
        try:
            self.progress_data['last_update'] = datetime.now().isoformat()
            
            def convert_types(obj):
                if isinstance(obj, dict):
                    return {k: convert_types(v) for k, v in obj.items()}
                elif isinstance(obj, list):
                    return [convert_types(v) for v in obj]
                elif hasattr(obj, 'item'):
                    return obj.item()
                elif hasattr(obj, 'tolist'):
                    return obj.tolist()
                else:
                    return obj
            
            clean_data = convert_types(self.progress_data)
            local_path = f"./{CONFIG['progress_file']}"
            
            with open(local_path, 'w') as f:
                json.dump(clean_data, f, indent=2)
            
            upload_file(
                path_or_fileobj=local_path,
                path_in_repo=CONFIG['progress_file'],
                repo_id=CONFIG['output_repository'],
                repo_type="dataset",
                commit_message=f"Corpus progress: {self.progress_data['total_qa_pairs_created']} QA pairs"
            )
            print(f"Progress saved to repository")
        except Exception as e:
            print(f"Failed to save progress: {e}")
    
    def is_chunk_processed(self, chunk_id):
        return chunk_id in self.progress_data['processed_chunks']
    
    def mark_chunk_processed(self, chunk_id, qa_count):
        if chunk_id not in self.progress_data['processed_chunks']:
            self.progress_data['processed_chunks'].append(chunk_id)
            self.progress_data['total_chunks_processed'] += 1
            self.progress_data['total_qa_pairs_created'] += qa_count

In [ ]:
# Cell 9: Deep Thinking Legal QA Generator
import random
import re

class DeepThinkingLegalQAGenerator:
    def __init__(self, progress_manager):
        safety_settings = [
            {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"},
        ]
        
        self.model = genai.GenerativeModel(
            CONFIG['model_name'],
            generation_config=genai.types.GenerationConfig(
                temperature=CONFIG['temperature'],
                max_output_tokens=CONFIG['max_output_tokens'],
                top_p=0.9,
                top_k=40
            ),
            system_instruction=SYSTEM_PROMPT,
            safety_settings=safety_settings
        )
        
        self.language_scorer = LanguageScorer(DOMAIN_CONFIG['target_language'])
        self.progress_manager = progress_manager
        self.request_count = progress_manager.progress_data['request_count']
        
        # Pre-compute combinations
        self.combinations = self.generate_all_combinations()
        self.combination_index = progress_manager.progress_data.get('combination_index', 0)
    
    def generate_all_combinations(self):
        """Generate all category + topic combinations"""
        combinations = []
        
        for category, topics in LEGAL_QA_TAXONOMY.items():
            for topic in topics:
                combinations.append({
                    'category': category,
                    'qa_topic': topic
                })
        
        random.shuffle(combinations)
        print(f"Total deep thinking QA combinations: {len(combinations)}")
        return combinations
    
    def get_next_combination(self):
        """Get next combination - fully random selection"""
        combo = random.choice(self.combinations)
        return combo
    
    def generate_deep_thinking_qa(self, combination, qa_id, retry_count=0):
        """Generate QA with deep iterative thinking"""
        prompt = DEEP_THINKING_QA_PROMPT.format(
            qa_topic=combination['qa_topic'],
            category=combination['category']
        )
        
        try:
            time.sleep(6)  # Longer wait for complex generation
            response = self.model.generate_content(prompt)
            self.request_count += 1
            self.progress_manager.progress_data['request_count'] = self.request_count
            
            if not response.candidates:
                if retry_count < 2:
                    return self.generate_alternative_deep_qa(combination, qa_id, retry_count + 1)
                return None
            
            candidate = response.candidates[0]
            
            if candidate.finish_reason == 2:  # Safety filter
                print(f"    QA {qa_id} filtered, trying alternative...")
                if retry_count < 2:
                    return self.generate_alternative_deep_qa(combination, qa_id, retry_count + 1)
                return None
            
            if candidate.finish_reason == 3:  # Truncated
                print(f"    QA {qa_id} truncated, getting continuation...")
                partial = candidate.content.parts[0].text.strip()
                continuation = self.get_thinking_continuation(combination, partial)
                qa_text = partial + "\n\n" + continuation if continuation else partial
            else:
                qa_text = candidate.content.parts[0].text.strip()
            
            # Parse and validate
            parsed = self.parse_deep_thinking_qa(qa_text)
            if parsed and self.validate_deep_thinking(parsed):
                return parsed
            else:
                print(f"    QA {qa_id} failed validation")
                if retry_count < 1:
                    return self.generate_deep_thinking_qa(combination, qa_id, retry_count + 1)
                return None
            
        except Exception as e:
            print(f"    Error: {e}")
            return None
    
    def parse_deep_thinking_qa(self, qa_text):
        """Parse question, thinking, and answer"""
        # Try to extract three parts
        q_match = re.search(r'Question:\s*(.+?)(?=Thinking:|$)', qa_text, re.DOTALL | re.IGNORECASE)
        t_match = re.search(r'Thinking:\s*(.+?)(?=Answer:|$)', qa_text, re.DOTALL | re.IGNORECASE)
        a_match = re.search(r'Answer:\s*(.+?)$', qa_text, re.DOTALL | re.IGNORECASE)
        
        if q_match and t_match and a_match:
            question = q_match.group(1).strip()
            thinking = t_match.group(1).strip()
            answer = a_match.group(1).strip()
            
            # Basic length validation
            if len(question) > 50 and len(thinking) > 300 and len(answer) > 50:
                return {
                    'question': question,
                    'thinking': thinking,
                    'answer': answer,
                    'full_text': qa_text
                }
        
        # Alternative: Try splitting by major sections
        sections = re.split(r'\n(?=Question:|Thinking:|Answer:)', qa_text, flags=re.IGNORECASE)
        
        q_text = ""
        t_text = ""
        a_text = ""
        
        for section in sections:
            if section.strip().lower().startswith('question'):
                q_text = re.sub(r'^question:?\s*', '', section.strip(), flags=re.IGNORECASE)
            elif section.strip().lower().startswith('thinking'):
                t_text = re.sub(r'^thinking:?\s*', '', section.strip(), flags=re.IGNORECASE)
            elif section.strip().lower().startswith('answer'):
                a_text = re.sub(r'^answer:?\s*', '', section.strip(), flags=re.IGNORECASE)
        
        if len(q_text) > 50 and len(t_text) > 300 and len(a_text) > 50:
            return {
                'question': q_text,
                'thinking': t_text,
                'answer': a_text,
                'full_text': qa_text
            }
        
        return None
    
    def validate_deep_thinking(self, parsed_qa):
        """Validate thinking depth and quality"""
        thinking = parsed_qa['thinking']
        answer = parsed_qa['answer']
        
        thinking_words = len(thinking.split())
        answer_words = len(answer.split())
        
        # Check minimum lengths
        if thinking_words < THINKING_REQUIREMENTS['min_thinking_words']:
            print(f"    Thinking too short: {thinking_words} words (need {THINKING_REQUIREMENTS['min_thinking_words']})")
            return False
        
        if answer_words < THINKING_REQUIREMENTS['min_answer_words']:
            print(f"    Answer too short: {answer_words} words")
            return False
        
        if answer_words > THINKING_REQUIREMENTS['max_answer_words']:
            print(f"    Answer too long: {answer_words} words (should be {THINKING_REQUIREMENTS['max_answer_words']} max)")
        
        # Check for iteration indicators
        iteration_indicators = [
            'iteration', 'reconsider', 'wait', 'actually', 'on second thought',
            'alternatively', 'however', 'but', 'yet', 'reconsidering',
            'let me check', 'verif', 'validat', 'cross-check'
        ]
        
        indicator_count = sum(1 for indicator in iteration_indicators if indicator in thinking.lower())
        
        if indicator_count < THINKING_REQUIREMENTS['required_iterations']:
            print(f"    Insufficient iteration indicators: {indicator_count} (need {THINKING_REQUIREMENTS['required_iterations']})")
            return False
        
        print(f"    Validated: Thinking={thinking_words}w, Answer={answer_words}w, Iterations={indicator_count}")
        return True
    
    def generate_alternative_deep_qa(self, combination, qa_id, retry_count):
        """Alternative generation with structured template"""
        prompt = ALTERNATIVE_DEEP_THINKING_PROMPT.format(
            qa_topic=combination['qa_topic']
        )
        
        try:
            time.sleep(5)
            response = self.model.generate_content(prompt)
            self.request_count += 1
            
            if response.candidates and response.candidates[0].finish_reason not in [2, 4]:
                qa_text = response.candidates[0].content.parts[0].text.strip()
                parsed = self.parse_deep_thinking_qa(qa_text)
                
                if parsed and self.validate_deep_thinking(parsed):
                    print(f"    Alternative generation successful")
                    return parsed
        except:
            pass
        
        return None
    
    def get_thinking_continuation(self, combination, partial):
        """Continue truncated thinking process"""
        continuation_prompt = f"""Continue this legal deep thinking analysis from where it was cut off:

{partial[-800:]}

Continue the iterative analysis and complete with the final answer section.

CONTINUATION:"""
        
        try:
            time.sleep(4)
            response = self.model.generate_content(continuation_prompt)
            if response.candidates and response.candidates[0].finish_reason not in [2, 4]:
                cont = response.candidates[0].content.parts[0].text.strip()
                print(f"    Got continuation: {len(cont)} chars")
                return cont
        except:
            pass
        
        return ""
    
    def generate_batch(self, batch_size):
        """Generate batch of deep thinking QA pairs"""
        results = []
        
        for i in range(batch_size):
            print(f"\nGenerating Deep Thinking QA {i+1}/{batch_size}")
            
            combo = self.get_next_combination()
            print(f"  Topic: {combo['qa_topic'][:60]}...")
            
            qa_pair = self.generate_deep_thinking_qa(combo, i)
            
            if not qa_pair:
                print(f"  Failed to generate QA {i+1}")
                continue
            
            # Language scoring
            language_scores = self.language_scorer.score_language_accuracy(
                qa_pair['question'],
                qa_pair['answer']
            )
            
            result = {
                'qa_id': f"deep_legal_qa_{self.progress_manager.progress_data['total_qa_pairs_created'] + len(results):06d}",
                'question': qa_pair['question'],
                'thinking': qa_pair['thinking'],
                'answer': qa_pair['answer'],
                
                # Lengths
                'question_length': len(qa_pair['question']),
                'thinking_length': len(qa_pair['thinking']),
                'answer_length': len(qa_pair['answer']),
                'question_word_count': len(qa_pair['question'].split()),
                'thinking_word_count': len(qa_pair['thinking'].split()),
                'answer_word_count': len(qa_pair['answer'].split()),
                'thinking_to_answer_ratio': len(qa_pair['thinking'].split()) / len(qa_pair['answer'].split()),
                
                # Metadata
                'qa_topic': combo['qa_topic'],
                'legal_category': combo['category'],
                'thinking_style': 'iterative',
                'combination_index': self.combination_index - 1,
                
                # Language
                **language_scores,
                
                'timestamp': datetime.now().isoformat(),
            }
            
            results.append(result)
            print(f"  ✓ Deep thinking QA complete: T/A ratio = {result['thinking_to_answer_ratio']:.1f}")
        
        return results

In [ ]:
# Cell 10: Deep Thinking QA Processing
def process_deep_thinking_qa_batch(target_count):
    """Generate batch of deep thinking legal QA"""
    generator = DeepThinkingLegalQAGenerator(progress_manager)
    
    already_generated = progress_manager.progress_data.get('total_qa_pairs_created', 0)
    remaining = target_count - already_generated
    
    if remaining <= 0:
        print(f"Target of {target_count} deep thinking QA pairs reached")
        return
    
    batch_size = min(remaining, CONFIG['batch_size'])
    
    print(f"Generating {batch_size} deep thinking legal QA ({already_generated}/{target_count} complete)")
    
    results = generator.generate_batch(batch_size)
    
    if results:
        save_deep_thinking_qa_batch(results, already_generated)
        
        progress_manager.progress_data['total_qa_pairs_created'] += len(results)
        progress_manager.save_progress()
        
        print(f"\nBatch complete: {len(results)} deep thinking QA pairs")

def save_deep_thinking_qa_batch(results, batch_start):
    """Save with deep thinking schema"""
    if not results:
        return
    
    try:
        df = pd.DataFrame(results)
        
        df = df.astype({
            'qa_id': 'string',
            'question': 'string',
            'thinking': 'string',
            'answer': 'string',
            'qa_topic': 'string',
            'legal_category': 'string',
        })
        
        batch_id = f"{batch_start:06d}_{batch_start+len(results):06d}"
        filename = f"train-{batch_id}.parquet"
        
        temp_filepath = f"/tmp/{filename}"
        df.to_parquet(temp_filepath, index=False)
        
        upload_file(
            path_or_fileobj=temp_filepath,
            path_in_repo=filename,
            repo_id=CONFIG['output_repository'],
            repo_type="dataset",
            commit_message=f"Deep thinking QA {batch_id}: {len(results)} pairs"
        )
        
        print(f"  ✓ Uploaded {filename}")
        os.remove(temp_filepath)
        
    except Exception as e:
        print(f"Failed to save: {e}")

def continue_deep_thinking_generation(target_count=2000):
    """Continue generating deep thinking QA"""
    while progress_manager.progress_data.get('total_qa_pairs_created', 0) < target_count:
        process_deep_thinking_qa_batch(target_count)

In [ ]:
# Cell 11: Utility Functions
def show_corpus_config():
    """Display current corpus configuration"""
    print("Corpus Synthesis Configuration:")
    print(f"  Domain: {DOMAIN_CONFIG['domain']}")
    print(f"  Approach: {DOMAIN_CONFIG['approach']}")
    print(f"  Source Dataset: {CONFIG['source_dataset']}")
    print(f"  Corpus Column: {CONFIG['corpus_column']}")
    print(f"  Variants per text: {DOMAIN_CONFIG['num_variants']}")
    print(f"  Max text length: {DOMAIN_CONFIG['max_text_length']}")

def update_corpus_config(corpus_column=None, approach=None, num_variants=None, max_text_length=None):
    """Update corpus configuration"""
    if corpus_column:
        CONFIG['corpus_column'] = corpus_column
    if approach:
        DOMAIN_CONFIG['approach'] = approach
    if num_variants:
        DOMAIN_CONFIG['num_variants'] = num_variants
    if max_text_length:
        DOMAIN_CONFIG['max_text_length'] = max_text_length
    
    print("Updated corpus configuration")
    show_corpus_config()

def show_progress():
    """Show current progress with enhanced file validation"""
    stats = progress_manager.progress_data
    print(f"Corpus Synthesis Progress:")
    print(f"  Current row: {stats.get('current_row', 0)}")
    print(f"  Processed rows: {len(stats.get('processed_rows', []))}")
    print(f"  QA pairs created: {stats['total_qa_pairs_created']}")
    print(f"  API requests: {stats['request_count']}")
    
    # Validate HF dataset compatibility
    validate_dataset_files()

# Keep the rest of Cell 11 as is...

## RUN

In [ ]:
# Initialize progress manager
progress_manager = ProgressManager()

print("Corpus-Based QA Synthesis Pipeline Ready!")
print("=" * 50)
print("Key Functions:")
print("- show_corpus_config() - Display current configuration")
print("- update_corpus_config(corpus_column, chunk_method, chunk_size, approach) - Update settings")
print("- continue_corpus_synthesis() - Start/continue synthesis")
print("- process_corpus_batch(start_row, end_row) - Process specific rows")
print("- show_progress() - Check current progress")
print("\nFirst, update CONFIG with your dataset info, then run: continue_corpus_synthesis()")

#show_corpus_config()
#continue_corpus_synthesis()
# Generate 2000 deep thinking legal QA pairs
continue_deep_thinking_generation(target_count=1000000)

## END